# Extracting Geospatial Embeddings with Clay Foundation Model
### Roma Norte, Mexico City

**Research internship (estancia de investigación) — Master's in Data Science, ITAM**
Student: Manuel Alonso De la Tejera González
Supervisor: Carlos López de la Cerda — Washington University in St. Louis

---

## Objective

This notebook documents the extraction of a geospatial embedding with **Clay v1.5**
for a Sentinel-2 image of Roma Norte, Mexico City, as a comparison point against
the **AlphaEarth (GSED)** embeddings used in the internship's main project,
*Earth Embedding Benchmarks for Geospatial Prediction*.

The relevant methodological difference is that AlphaEarth provides 64-dimensional
embeddings **precomputed** at the tile level (10m, annual, 2017–2024) directly from
Google Earth Engine, while Clay generates **1024-dimensional** embeddings from
Sentinel-2 imagery processed on the fly, through a Vision Transformer encoder
trained with a *Masked Autoencoder* (MAE) objective. That difference in
computational cost and dimensionality matters when deciding which one to use in
the internship's real-estate valuation pipeline.

## References

- Clay Foundation (2024). *Clay Foundation Model*. https://clay-foundation.github.io/model/
- Brown et al. (2025). *AlphaEarth Foundations*. arXiv:2507.22291
- Klemmer et al. (2025). *Earth Embeddings: Towards AI-centric representations of our planet*.


## 1. Environment setup

We clone the official Clay repository (needed for the `claymodel` package and
for `configs/metadata.yaml`, which we use later for normalization) and install the
additional dependencies for Google Earth Engine and Hugging Face Hub.

In [1]:
# Clone the official Clay repository
!git clone https://github.com/Clay-foundation/model.git
%cd model

# Install the package and additional dependencies
!pip install -e . -q
!pip install earthengine-api huggingface_hub -q

Cloning into 'model'...
remote: Enumerating objects: 2959, done.
remote: Counting objects: 100% (1430/1430), done.
remote: Compressing objects: 100% (611/611), done.
remote: Total 2959 (delta 1091), reused 974 (delta 805), pack-reused 1529 (from 3)
Receiving objects: 100% (2959/2959), 212.96 MiB | 20.11 MiB/s, done.
Resolving deltas: 100% (1811/1811), done.
/content/model
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━

### Google Earth Engine authentication

In [2]:
import ee
from google.colab import auth

auth.authenticate_user()
ee.Authenticate()
ee.Initialize(project='earth-embeddings-project')

print("GEE authenticated successfully")

GEE authenticated successfully


### Hardware check

Clay v1.5 has ~633M parameters; running this notebook on a GPU is recommended
(on Colab, a T4 runtime or better).

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 2. Loading the Clay v1.5 model

We download the pretrained weights from Hugging Face Hub and load the model in
evaluation mode.

In [4]:
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(
    repo_id="made-with-clay/Clay",
    filename="v1.5/clay-v1.5.ckpt"
)
print(f"Weights downloaded to: {weights_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


v1.5/clay-v1.5.ckpt:   0%|          | 0.00/5.16G [00:00<?, ?B/s]

Weights downloaded to: /root/.cache/huggingface/hub/models--made-with-clay--Clay/snapshots/70200ebcccdf67bf2a0cb9984c77ddee26c10ed2/v1.5/clay-v1.5.ckpt


In [5]:
import torch
import sys
sys.path.append('/content/model')

from claymodel.module import ClayMAEModule

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = ClayMAEModule.load_from_checkpoint(
    weights_path,
    metadata_path='/content/model/configs/metadata.yaml',
    map_location=device
)
model.eval()
print(f"Model loaded on: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Model loaded on: cuda
Total parameters: 632,763,392


## 3. Downloading the satellite image

We define the area of interest (Roma Norte, CDMX) and download a 2022 Sentinel-2
image with low cloud coverage.

Two corrections compared to the first attempt:

1. **Download format.** `getDownloadURL` with `format='NPY'` flattens all bands
   into a single 2D array, losing the channel dimension. We use
   `ZIPPED_GEO_TIFF_PER_BAND` instead, which exports each band as a separate
   GeoTIFF and lets us reconstruct the `(bands, height, width)` array that Clay
   expects.
2. **Band order.** `configs/metadata.yaml` defines the official `band_order` for
   `sentinel-2-l2a` as `blue, green, red, rededge1, rededge2, rededge3, nir,
   nir08, swir16, swir22` — i.e. `B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12`. We
   keep that order so it stays consistent with the per-band normalization in the
   next section.

In [6]:
import ee
import numpy as np
import requests, io, zipfile
import rasterio

# Area of interest — Roma Norte, CDMX
aoi = ee.Geometry.Rectangle([-99.165, 19.410, -99.150, 19.425])

# Sentinel-2 image with low cloud coverage, 2022
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate('2022-01-01', '2022-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .sort('CLOUDY_PIXEL_PERCENTAGE') \
    .first()

# Sentinel-2 optical bands Clay expects, in the official order from
# configs/metadata.yaml (sentinel-2-l2a band_order)
bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
s2_clipped = s2.select(bands).clip(aoi)

url = s2_clipped.getDownloadURL({
    'scale': 10,
    'region': aoi,
    'format': 'ZIPPED_GEO_TIFF_PER_BAND',
    'bands': bands
})

response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))

arrays = []
for band in bands:
    fname = [f for f in z.namelist() if band in f][0]
    with rasterio.open(io.BytesIO(z.read(fname))) as src:
        arrays.append(src.read(1))

chip = np.stack(arrays, axis=0).astype(np.float32)
print(f"Chip shape: {chip.shape}")  # expected: (10, H, W)
print(f"Min: {chip.min():.1f}, Max: {chip.max():.1f}")

Chip shape: (10, 167, 159)
Min: 74.0, Max: 5402.0


In [23]:
patch_size = 8
size = (min(chip.shape[1], chip.shape[2]) // patch_size) * patch_size
top = (chip.shape[1] - size) // 2
left = (chip.shape[2] - size) // 2
chip = chip[:, top:top+size, left:left+size]

print(f"Cropped chip shape: {chip.shape}")  # debe quedar cuadrado, p. ej. (10, 152, 152)

Cropped chip shape: (10, 152, 152)


## 4. Preprocessing: per-band normalization

Clay does not use a generic normalization like dividing by 10000; each band has
its own mean and standard deviation, computed over the training data and reported
in `configs/metadata.yaml`. We standardize each channel with those values so the
image reaches the encoder in the same distribution the model was trained on.

In [25]:
import yaml
import torch

with open('/content/model/configs/metadata.yaml', 'r') as f:
    metadata = yaml.safe_load(f)

sensor_meta = metadata['sentinel-2-l2a']
# same order as `bands` in the previous section
band_names = ['blue', 'green', 'red', 'rededge1', 'rededge2', 'rededge3',
              'nir', 'nir08', 'swir16', 'swir22']

means = torch.tensor([sensor_meta['bands']['mean'][b] for b in band_names]).view(-1, 1, 1)
stds  = torch.tensor([sensor_meta['bands']['std'][b]  for b in band_names]).view(-1, 1, 1)

chip_t = torch.from_numpy(chip).float()          # (10, H, W)
chip_norm = (chip_t - means) / stds              # per-band standardization

# Clay's encoder.forward unpacks pixels as B, C, H, W = cube.shape (4 dims) —
# there is no separate time axis on the image tensor itself. Date and location
# are passed as metadata in Section 5 instead.
chip_tensor = chip_norm.unsqueeze(0)              # (1, 10, H, W)

print(f"Tensor shape: {chip_tensor.shape}")
print(f"Min: {chip_tensor.min():.3f}, Max: {chip_tensor.max():.3f}")

Tensor shape: torch.Size([1, 10, 152, 152])
Min: -1.645, Max: 1.788


## 5. Spatial and temporal metadata

Clay does not just take the image: it conditions the embedding on the geographic
position and the acquisition date, encoded with sine/cosine functions bounded in
`[-1, 1]`, consistent with the rest of the model's inputs.

It is important to use exactly the scheme Clay documents — week of the year and
hour of the day for time, latitude/longitude **in radians** for space — because
the model never saw any other kind of encoding during training. Passing, for
example, raw latitude/longitude in degrees produces an embedding that is valid in
shape but not in content: the model receives values outside the range it learned
to interpret on those channels, and the resulting embedding stops being comparable
to one extracted from any other correctly processed chip.

The functions below replicate `normalize_timestamp` and `normalize_latlon` exactly
as documented in Clay's official repository.

In [26]:
import math
from datetime import datetime

def normalize_timestamp(date):
    """Position within the week of the year and the hour of the day, in sine/cosine."""
    week = date.isocalendar().week * 2 * math.pi / 52
    hour = date.hour * 2 * math.pi / 24
    return (math.sin(week), math.cos(week)), (math.sin(hour), math.cos(hour))

def normalize_latlon(lat, lon):
    """Latitude/longitude in radians, in sine/cosine."""
    lat_r = lat * math.pi / 180
    lon_r = lon * math.pi / 180
    return (math.sin(lat_r), math.cos(lat_r)), (math.sin(lon_r), math.cos(lon_r))

# Roma Norte, CDMX — center of the area of interest
lat, lon = 19.4175, -99.1575
# Representative date for the selected Sentinel-2 image (2022, cloud-free)
acquisition_date = datetime(2022, 6, 15)

week_sc, hour_sc = normalize_timestamp(acquisition_date)
time_encoding = torch.tensor([[*week_sc, *hour_sc]]).float()      # [1, 4]

lat_sc, lon_sc = normalize_latlon(lat, lon)
latlon_encoding = torch.tensor([[*lat_sc, *lon_sc]]).float()      # [1, 4]

# Wavelengths in micrometers, same order as the chip's bands
wavelengths = torch.tensor([
    0.493, 0.560, 0.665, 0.704, 0.740, 0.783, 0.842, 0.865, 1.610, 2.190
]).float()

# Ground Sampling Distance in meters
gsd = torch.tensor([10.0]).float()  # (1,)

print(f"time_encoding:   {time_encoding.shape}")    # [1, 4]
print(f"latlon_encoding: {latlon_encoding.shape}")   # [1, 4]
print(f"wavelengths:     {wavelengths.shape}")       # [1, 10]

time_encoding:   torch.Size([1, 4])
latlon_encoding: torch.Size([1, 4])
wavelengths:     torch.Size([10])


## 6. Checking the encoder architecture

Before building the input `datacube`, we confirm the dimensions the encoder
expects internally.

In [27]:
print("patch_size:", model.model.encoder.patch_size)
print("dim:", model.model.encoder.dim)

patch_size: 8
dim: 1024


To confirm the exact format the encoder uses to combine time and position,
we inspect the source of `add_encodings`. The key line is
`torch.hstack((time, latlon))`, which requires `time` and `latlon` to add up to
8 columns in total — hence the 4-column vectors built in the previous section.

In [28]:
import inspect
from claymodel.model import Encoder

print(inspect.getsource(model.model.encoder.add_encodings))

    def add_encodings(self, patches, time, latlon, gsd):
        """Add position encoding to the patches"""
        B, L, D = patches.shape

        grid_size = int(math.sqrt(L))
        self.num_patches = grid_size**2

        pos_encoding = (
            posemb_sincos_2d_with_gsd(
                h=grid_size,
                w=grid_size,
                dim=(self.dim - 8),
                gsd=gsd,
            )
            .to(patches.device)
            .detach()
        )  # [L (D - 8)]

        time_latlon = torch.hstack((time, latlon)).to(patches.device).detach()  # [B 8]

        pos_encoding = repeat(pos_encoding, "L D -> B L D", B=B)  # [B L (D - 8)]
        time_latlon = repeat(time_latlon, "B D -> B L D", L=L)  # [B L 8]
        pos_metadata_encoding = torch.cat(
            (pos_encoding, time_latlon), dim=-1
        )  # [B L D]

        patches = patches + pos_metadata_encoding  # [B L D] + [B L D] -> [B L D]
        return patches  # [B L D]



## 7. Extracting the embedding

With the image normalized and the metadata correctly encoded, we run the encoder
in inference mode (no gradient) and extract the embedding corresponding to the
class token (position 0), which summarizes the chip's global representation.

In [29]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

with torch.no_grad():
    output_tuple = model.model.encoder(
        datacube={
            'pixels': chip_tensor.to(device),
            'time':   time_encoding.to(device),
            'latlon': latlon_encoding.to(device),
            'waves':  wavelengths.to(device),
            'gsd':    gsd.to(device),
        }
    )

encoded_patches = output_tuple[0]
embedding = encoded_patches[:, 0, :].squeeze().cpu().numpy()

print(f"Embedding shape: {embedding.shape}")  # expected: (1024,)
print("First 5 dimensions:")
for i in range(5):
    print(f"  dim_{i:03d}: {embedding[i]:.6f}")

Embedding shape: (1024,)
First 5 dimensions:
  dim_000: 0.254732
  dim_001: 0.033040
  dim_002: 0.216200
  dim_003: -0.578948
  dim_004: 0.240723


## 8. Discussion and next steps

The resulting embedding has **1024 dimensions**, compared to AlphaEarth/GSED's 64
dimensions. For the internship, this has two direct implications:

- **Dimensionality.** Any incremental comparison (Model A: census / Model B:
  embeddings / Model C: both) should treat dimensionality as an explicit factor —
  a 1024-dimensional vector does not automatically "outperform" a 64-dimensional
  one; it is worth evaluating with regularization (e.g. PCA or Lasso) before
  comparing predictive power.
- **Computational cost.** AlphaEarth already provides a precomputed vector per
  tile from GEE; Clay requires downloading the Sentinel-2 image and running the
  encoder for every point or AGEB, which is considerably more expensive at the
  scale of all of Mexico City.

This notebook processes a single point with a single date. To integrate it into
the internship's pipeline, the remaining steps are: (1) decide the date or time
window to use per AGEB, (2) automate the download and embedding computation for
every AGEB in Mexico City, and (3) decide whether to use the average embedding
across several chips per AGEB or a single chip centered on the centroid.